# 04 · Adu versi penguatan INATRADE (beras)

Urutan: **coba → penyetelan → kalibrasi → ujian akhir (U1)**. Rancangan dan aturan: `binpas/RANCANGAN_INATRADE.md`
(bagian 4 dan 7); ambang lolos yang sudah dikunci: `binpas/ambang_ujian.md`.

| Langkah | Blok data | Isi | Perkiraan waktu |
| --- | --- | --- | --- |
| Coba | penyetelan | uji kode dengan sedikit ulangan; hasilnya tidak dipakai | ±5–10 menit |
| Penyetelan | Jan 2024–Jun 2025 | adu 79 versi bertahap (50 → 200 → 500 ulangan), pilih resep | ±1–3 jam di Colab |
| Kalibrasi | Jul–Des 2025 | bias dan pelebar rentang (konformal) untuk resep | ±5–20 menit |
| **Ujian akhir U1** | Jan–Sep 2026 | **sekali jalan**; hasilnya yang dilaporkan | ±5–20 menit |

Aturan yang harus dijaga:
- Jangan ubah kode, ambang, atau resep setelah melihat hasil kalibrasi/ujian.
- Sel **U1 hanya boleh dijalankan sekali** (skrip menolak jalan kedua). Menjalankan ulang setelah melihat hasil = ujian tidak sah.
- Bila sesi Colab terputus saat penyetelan, jalankan ulang sel penyetelan: tahap yang sudah selesai dilewati (`status.json`).
- Data INATRADE internal: jangan bagikan folder ini.

In [ ]:
import os, sys
DI_COLAB = 'google.colab' in sys.modules
if DI_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/inatrade_colab'   # <- ganti bila folder diunggah ke tempat lain
else:
    BASE = os.path.abspath('.')
os.chdir(BASE)
XLSM = 'data/inatrade/Non_PAW_dari_nol.xlsm'
BPS = 'data/bps'
assert os.path.exists(XLSM), f'file INATRADE tidak ditemukan: {BASE}/{XLSM}'
print('folder kerja:', BASE, '| CPU:', os.cpu_count())

OUT = 'hasil_adu_beras'      # semua keluaran adu versi (di Drive, jadi tidak hilang bila sesi terputus)
!pip install -q catboost quantile-forest statsmodels

## 1. Coba (uji kode)
Semua langkah dijalankan dengan sedikit ulangan **di blok penyetelan saja**; blok kalibrasi dan ujian tidak disentuh. Keluaran ke folder terpisah.

In [ ]:
!python binpas/adu_versi.py coba "$XLSM" hasil_adu_coba "$BPS"

## 2. Penyetelan (lama)
Biarkan tab tetap terbuka. Bila terputus, jalankan ulang sel ini.

In [ ]:
!python binpas/adu_versi.py penyetelan "$XLSM" "$OUT" "$BPS"

In [ ]:
import json, pandas as pd
print(json.dumps(json.load(open(f'{OUT}/resep.json')), indent=1))
display(pd.read_csv(f'{OUT}/gerbang_penyetelan.csv').T)

## 3. Kalibrasi (Jul–Des 2025)
Resep sudah terkunci di `resep.json`. Langkah ini hanya menghitung bias dan pelebar rentang.

In [ ]:
!python binpas/adu_versi.py kalibrasi "$XLSM" "$OUT" "$BPS"

## 4. Ujian akhir U1 (Jan–Sep 2026) — SEKALI JALAN
Sebelum menjalankan: pastikan `resep.json`, `kalibrasi.pkl`, kode, dan `binpas/ambang_ujian.md` tidak akan diubah lagi.
Ubah `SAYA_YAKIN` menjadi `True` lalu jalankan.

In [ ]:
SAYA_YAKIN = False
if SAYA_YAKIN:
    !python binpas/adu_versi.py ujian "$XLSM" "$OUT" "$BPS"
else:
    print('U1 belum dijalankan. Ubah SAYA_YAKIN = True bila sudah siap.')

## 5. Baca hasil U1
Kolom `L_...` = lolos/tidak tiap gerbang; `cakupan_belum_bisa` = data terlalu sedikit untuk memutuskan cakupan.
Tangga klaim: laporkan tingkat terhalus yang lolos semua gerbang (prov → wil → nas). Bila tidak ada yang lolos: pakai data asli berlabel "tercatat".

In [ ]:
import pandas as pd
g = pd.read_csv(f'{OUT}/gerbang_U1.csv')
display(g.T)
lolos = g[g.L_lebih_baik & g.L_konsisten & g.L_cakupan & g.L_bias & (g.arah >= 0.8)]
print('tingkat yang lolos semua gerbang:', list(lolos.tingkat) or 'tidak ada')